# 02 — Run the full recoverability grid

> ⚠️ **This notebook overwrites data.** `experiment.run_all()` re-simulates every scenario
> and rewrites **all three data tiers** — Bronze, Silver and Gold. It takes several minutes.
> If you only want to rebuild the Gold tables from the existing summary, do **not** run this
> notebook; run `python experiment.py tables` from `src/` instead, which re-derives the
> tables without re-simulating anything.

**Purpose.** Execute the whole study: every `(p, kappa)` cell, every replicate, every
estimator variant, and aggregate the result into the summary that every figure and table
reads from.

**Inputs.** `src/config.py` only — the grid, the seed and the thresholds. There is no
external data. Everything downstream is generated deterministically from `BASE_SEED = 42`,
which is what makes the study reproducible from a clean clone.

**Outputs.**

| Tier | File | Grain |
|---|---|---|
| Bronze | `data/1_bronze/bronze_synthetic_days.parquet` | one row per simulated day (gitignored, regenerable) |
| Silver | `data/2_silver/silver_estimates.csv` | one row per scenario × replicate × estimator |
| Gold | `data/3_gold/tables/recoverability_summary.csv` | one row per `(p, kappa)` × estimator, replicates averaged |
| Gold | `data/3_gold/tables/table3_headline_results.csv`, `table4_win_counts.csv`, `experiment_factors.csv` | the paper's tables |

**Process.**

1. Put `src/` on the path and import `config` and `experiment`.
2. Run the grid and write all three tiers.
3. Inspect the Gold summary.

## Setup

Only `config` and `experiment` are needed — `experiment` imports `synth`, `estimators` and
`metrics` itself, so this notebook orchestrates rather than reimplements.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import experiment

## Run the full grid

**The shape of the run.** 10 values of `p` × 10 values of `kappa` × 20 replicates = 2,000
simulated years, each scored by 5 estimator variants, giving **10,000 Silver rows**. Each of
those years is 365 days, so Bronze holds 730,000 rows.

**Why replicates.** One year at one `(p, kappa)` is a single draw, and a lucky or unlucky
draw would be indistinguishable from a good or bad estimator. Twenty replicates per cell let
the Gold aggregation report a mean *and* a standard deviation, so dispersion is visible
rather than assumed away.

**Seeding.** `experiment._seed_for` computes
`BASE_SEED + p_idx * 100_000 + k_idx * 1_000 + replicate`. The strides are far wider than
the ranges they encode, so no two scenarios can ever collide on a seed — every simulated
year is independent and exactly reproducible. The mixture fit is seeded separately at
`seed + 1`, keeping the estimator's own randomness decoupled from the data's.

**What each estimator gets.** All five see the same `load` vector. Only the oracle variant
is passed the true `p` and `kappa`; the estimated variant recovers them from the data. That
separation is the study's integrity condition — any leak into the estimated path would
invalidate the result.

In [ ]:
experiment.run_all()

## Inspect the Gold summary

`recoverability_summary.csv` is 500 rows — one per `(p, kappa)` cell per estimator, with the
20 replicates collapsed into `r_f_mean`, `r_f_std`, `mae_b_mean`, `mae_b_std` and
`fallback_weight_mean`. **Every figure and every table in the paper reads from this file**,
which is what keeps the manuscript and the code from drifting apart.

Two columns worth understanding before reading any result:

- **`r_f_mean`** is the flexible-energy recovery ratio, `E_F_hat / E_F_true`. A value of 1
  is perfect. It runs very high in the sparse-event corner, and that is *not* the
  estimator's fault — `F_hat = max(L - B_hat, 0)` keeps only the positive half of the
  noise, so even a flawless backbone books `0.399 * sigma` of phantom flexibility on every
  quiet day. `config.NOISE_FLOOR_PER_DAY` holds that constant, and the closed form is
  `R_F ≈ 1 + (1-p) * 0.399 / (p * kappa)`.
- **`mae_b_mean`** is the backbone error, which the noise floor does not contaminate at all.
  It is the cleaner measure of whether the quantile was chosen well.

- **`fallback_weight_mean`** is populated only for `estimated_aqf` — the other four variants
  fit no mixture, so it is empty for them.

In [ ]:
import pandas as pd

gold = pd.read_csv(config.GOLD_TABLES_DIR / "recoverability_summary.csv")
gold.head()

## Conclusion

All three data tiers are now written and the study is complete as data. The Gold summary is
the single artefact everything downstream depends on: no figure, table or quoted number
should ever be computed anywhere but from `data/3_gold/tables/`.

Because the run is fully seeded, re-executing this notebook reproduces every number
exactly — the results have been checked to hold across Python 3.11/scikit-learn 1.8 and
3.12/scikit-learn 1.9.

Next: notebook 03 turns this summary into the paper's figures.